In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# PLotting the Buzi River and smaller river (Gauge 3). Supplementary Figure 1.3. 
# Script developed by Poppy Webb. 

# --- File paths ---
factual = r"/Data/Scenario_floodmaps/event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0_lisboa_2020/sfincs.dis"
file_counterfactual = r"/Data/Scenario_floodmaps/event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0_lisboa_2000/sfincs.dis"

# --- Load discharge files (.dis is whitespace-delimited ASCII) ---
df_2020 = pd.read_csv(factual, delim_whitespace=True, header=None)
df_counter = pd.read_csv(file_counterfactual, delim_whitespace=True, header=None)

# Name columns: time_s + Q1..QN
ncols = df_2020.shape[1]
if ncols < 2:
    raise ValueError(f"Expected at least 2 columns (time + discharge). Got {ncols} columns.")
nQ = ncols - 1
colnames = ["time_s"] + [f"Q{i+1}" for i in range(nQ)]
df_2020.columns = colnames
df_counter.columns = colnames

# --- Event start + desired window ---
event_start = pd.Timestamp("2019-03-11 00:00")
plot_start = pd.Timestamp("2019-03-11")
plot_end   = pd.Timestamp("2019-03-25")

def add_time_index(df: pd.DataFrame, start: pd.Timestamp) -> pd.DataFrame:
    df = df.copy()
    df["time"] = start + pd.to_timedelta(df["time_s"], unit="s")
    return df.set_index("time")

df_2020_t = add_time_index(df_2020, event_start)
df_counter_t = add_time_index(df_counter, event_start)

# --- Gauges to plot (index -> label); Q column assumed Q{idx+1} ---
gauge_map = [
    (0, "Buzi river gauge"),   # Q1
    (2, "Small river Gauge"),  # Q3 (change index if needed)
]
panel_labels = ["(a)", "(b)"]

# --- One figure, stacked panels (less long + less thin) ---
fig, axes = plt.subplots(2, 1, figsize=(10, 7.5), dpi=300, sharex=True)
fig.subplots_adjust(hspace=0.18)

# Date ticks (shared)
locator = mdates.DayLocator(interval=3)
formatter = mdates.DateFormatter("%d-%m-%Y")

for ax, (idx, gauge_name), lab in zip(axes, gauge_map, panel_labels):
    qcol = f"Q{idx+1}"
    if qcol not in df_2020_t.columns:
        raise ValueError(f"{qcol} not found. Available: Q1..Q{nQ}. Check gauge_map indices.")

    # Plot series
    ax.plot(df_2020_t.index, df_2020_t[qcol], label="Factual (2020)", linewidth=2)
    ax.plot(df_counter_t.index, df_counter_t[qcol], linestyle="--",
            label="Counterfactual (2000 LULC)", linewidth=2)

    # Titles (gauge names)
    ax.set_title(gauge_name, fontsize=12, pad=6)

    # Panel label (top-left)
    ax.text(
        0.01, 0.98, lab,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12, fontweight="bold"
    )

    # Legend inside but not overlapping panel label
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.02, 0.88),  # move legend slightly down
        fontsize=9,
        frameon=True,
        borderaxespad=0.0
    )

    # Axes formatting
    ax.set_xlim(plot_start, plot_end)
    ax.set_ylabel("Discharge [m³/s]", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="y", labelsize=10)

    # Date formatting
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)
    ax.tick_params(axis="x", labelsize=9, labelbottom=True)


fig.tight_layout()
plt.show()

